In [ ]:
# Imports
import os
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import shap
import joblib

sys.path.append(os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', '3_utils'))

from data_loader import load_regional_data
from preprocessing import preprocess_lag_features

In [ ]:
# Global plot style
matplotlib.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 13,
    'axes.labelweight': 'medium',
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

In [ ]:
# Paths
SELECTION_DIR = os.path.join('..', '4_model_training_and_evaluation', '02_gpr', 'regional_level_tuning', 'model_selection_results')
SAVE_DIR = 'plots'
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Manually selected best regional GPR model after reviewing top 10 candidates
selected_filename = "trial_001.pkl"

In [ ]:
# Load selected model info from model selection results
csv_path = os.path.join(SELECTION_DIR, 'regional', 'regional_gpr_top_models_test_eval.csv')
df_results = pd.read_csv(csv_path)
row = df_results[df_results['model_filename'] == selected_filename].iloc[0]

model_path = row['model_path']
fire_lag = int(row['fire_lag'])
climate_lag = int(row['climate_lag'])
print(f"Loaded: {selected_filename} — fire_lag={fire_lag}, climate_lag={climate_lag}")

In [ ]:
# Helper — generate concise feature names from lag params
def generate_feature_names(fire_lag, climate_lag):
    climate_features = ['Temp', 'Precip', 'Hum', 'Wind']
    lagged_fire = [f'Fire_lag{i}' for i in range(1, fire_lag + 1)]
    lagged_climate = [f'{col}_lag{i}' for col in climate_features for i in range(1, climate_lag + 1)]
    return climate_features + lagged_fire + lagged_climate

In [ ]:
# Load model and data
model = joblib.load(model_path)
params = {'fire_lag': fire_lag, 'climate_lag': climate_lag}
df = load_regional_data()
X_train, _, _, _, X_test, _, df_test = preprocess_lag_features(df.copy(), params)
feature_names = generate_feature_names(fire_lag, climate_lag)

In [ ]:
# Extract GPR length scales and rank features by sensitivity
length_scales = model.kernel_.k2.length_scale
print(f"Total features with ARD: {len(length_scales)}")

feature_importance = sorted(
    zip(feature_names[:len(length_scales)], length_scales),
    key=lambda x: x[1]
)
top_15 = feature_importance[:15]
features, scales = zip(*top_15)

In [ ]:
# Compute SHAP values using kernel explainer on a small sample
X_train_sample = X_train[:400]
X_test_sample = X_test[:400]

explainer = shap.KernelExplainer(model.predict, X_train_sample)
shap_values = explainer.shap_values(X_test_sample, nsamples=50)

In [ ]:
# Plot length scales and SHAP summary side by side
top15_names = [f for f, _ in top_15]
top15_indices = [feature_names.index(f) for f in top15_names]
shap_values_top15 = np.array(shap_values)[:, top15_indices]
X_test_top15 = X_test_sample[:, top15_indices]

fig, axes = plt.subplots(1, 2, figsize=(34, 16))
colors = plt.cm.viridis(np.linspace(0, 1, len(features)))

axes[0].barh(features[::-1], scales[::-1], color=colors[::-1])
axes[0].set_xlabel("Length Scale (smaller = higher sensitivity)", labelpad=13)
axes[0].set_ylabel("Features", labelpad=13)
axes[0].tick_params(axis='y', pad=10)
for label in axes[0].get_yticklabels():
    label.set_fontweight('medium')
axes[0].grid(alpha=0.2, linestyle='--', linewidth=0.5)
axes[0].text(-0.1, 1.03, "(a)", transform=axes[0].transAxes,
             fontsize=15, fontweight='bold', va='bottom', ha='right')

plt.sca(axes[1])
shap.summary_plot(
    shap_values_top15,
    X_test_top15,
    feature_names=top15_names,
    show=False,
    plot_size=(10, 7),
    alpha=1.0
)

ax = plt.gca()
for text in ax.get_yticklabels() + ax.get_xticklabels():
    text.set_fontname('DejaVu Serif')
    text.set_fontsize(10)
    text.set_fontweight('medium')
for label in [ax.xaxis.label, ax.yaxis.label]:
    label.set_fontname('DejaVu Serif')
ax.xaxis.label.set_fontsize(13)
ax.tick_params(axis='y', pad=10)
ax.set_xlabel("SHAP value (impact on model output)", labelpad=15)
ax.text(-0.1, 1.03, "(b)", transform=ax.transAxes,
        fontsize=15, fontweight='bold', va='bottom', ha='right')

plt.tight_layout(pad=5.0)

out_path = os.path.join(SAVE_DIR, "gpr_regional_length_scales_shap.png")
plt.savefig(out_path, dpi=600, bbox_inches='tight')
plt.close()
print(f"Saved plot to {out_path}")